In [ ]:
# ============================================================
# 2. IMPORTS
# ============================================================

import os
import json
import pandas as pd

from huggingface_hub import HfApi, hf_hub_download


# ============================================================
# 3. HUGGING FACE TOKEN
# ============================================================

HF_TOKEN = ""


# ============================================================
# 4. CONFIGURATION
# ============================================================

REPO_ID = "IRIIS-RESEARCH/Nepali-Text-Corpus"

OUTPUT_DIR = "/kaggle/working/nepali_json"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 5. HUGGING FACE API
# ============================================================

api = HfApi(
    token=HF_TOKEN if HF_TOKEN.strip() else None
)


# ============================================================
# 6. FIND ALL PARQUET FILES
# ============================================================

print("Finding all Parquet files...")

all_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

parquet_files = sorted([
    file
    for file in all_files
    if file.startswith("data/")
    and file.endswith(".parquet")
])

print("\nTotal Parquet files:", len(parquet_files))


# ============================================================
# 7. GLOBAL STATISTICS
# ============================================================

total_files = 0
total_rows = 0
total_words = 0


# ============================================================
# 8. PROCESS EVERY PARQUET FILE
# ============================================================

for file_number, parquet_file in enumerate(
    parquet_files,
    start=1
):

    print("\n")
    print("=" * 75)
    print(
        f"PROCESSING FILE {file_number}/{len(parquet_files)}"
    )
    print("=" * 75)

    parquet_filename = os.path.basename(parquet_file)

    print("Parquet file:", parquet_filename)

    file_path = None

    try:

        # ====================================================
        # DOWNLOAD ONE PARQUET FILE
        # ====================================================

        file_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=parquet_file,
            repo_type="dataset",
            token=HF_TOKEN if HF_TOKEN.strip() else None
        )

        print("Downloaded successfully.")


        # ====================================================
        # READ PARQUET
        # ====================================================

        df = pd.read_parquet(file_path)

        print("Rows in Parquet:", len(df))
        print("Columns:", df.columns.tolist())


        # ====================================================
        # CHECK ARTICLE COLUMN
        # ====================================================

        if "Article" not in df.columns:

            print("ERROR: Article column not found.")
            continue


        # ====================================================
        # CREATE JSON FILE
        # ====================================================

        json_filename = (
            os.path.splitext(parquet_filename)[0]
            + ".json"
        )

        json_path = os.path.join(
            OUTPUT_DIR,
            json_filename
        )


        # ====================================================
        # STATISTICS FOR THIS PARQUET
        # ====================================================

        file_rows = 0
        file_words = 0


        # ====================================================
        # WRITE JSON
        # ====================================================
        #
        # IMPORTANT:
        #
        # Every row becomes ONE separate JSON object.
        #
        # There is NO duplicate removal.
        #
        # If there are 123456 rows, there will be
        # exactly 123456 JSON objects.
        #

        with open(
            json_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write("[\n")

            first_entry = True

            for article in df["Article"]:

                # ------------------------------------------------
                # Handle missing Article
                # ------------------------------------------------

                if pd.isna(article):
                    text = ""
                else:
                    text = str(article).strip()


                # ------------------------------------------------
                # Word count
                # ------------------------------------------------

                wordcount = len(text.split())


                # ------------------------------------------------
                # Update file statistics
                # ------------------------------------------------

                file_rows += 1
                file_words += wordcount


                # ------------------------------------------------
                # Create ONE JSON object for this row
                # ------------------------------------------------

                record = {
                    "source": "nepali-text-corpus",
                    "url": parquet_filename,
                    "text": text,
                    "wordcount": wordcount
                }


                # ------------------------------------------------
                # Write this individual JSON object
                # ------------------------------------------------

                if not first_entry:
                    f.write(",\n")

                json.dump(
                    record,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

                first_entry = False


            f.write("\n]")


        # ====================================================
        # UPDATE GLOBAL STATISTICS
        # ====================================================

        total_files += 1
        total_rows += file_rows
        total_words += file_words


        # ====================================================
        # DISPLAY THIS PARQUET'S STATISTICS
        # ====================================================

        print("\nFILE COMPLETED")
        print("-" * 60)

        print(
            "Parquet file:",
            parquet_filename
        )

        print(
            "Rows:",
            file_rows
        )

        print(
            "JSON entries:",
            file_rows
        )

        print(
            "Word count in this Parquet:",
            file_words
        )

        print(
            "JSON saved:",
            json_path
        )


        # ====================================================
        # REMOVE DOWNLOADED PARQUET
        # ====================================================

        if file_path and os.path.exists(file_path):

            os.remove(file_path)

            print(
                "Parquet removed from Kaggle disk."
            )


        # ====================================================
        # FREE MEMORY
        # ====================================================

        del df


    except Exception as e:

        print("\nERROR PROCESSING FILE")
        print(parquet_filename)

        print("Error:", e)

        print(
            "Continuing with next Parquet..."
        )

        continue


# ============================================================
# 9. FINAL CORPUS STATISTICS
# ============================================================

print("\n")
print("=" * 75)
print("CORPUS EXTRACTION COMPLETE")
print("=" * 75)

print(
    "Total Parquet files processed:",
    total_files
)

print(
    "Total JSON entries:",
    total_rows
)

print(
    "TOTAL WORD COUNT:",
    total_words
)

if total_rows > 0:

    print(
        "Average words per article:",
        round(
            total_words / total_rows,
            2
        )
    )

print(
    "\nJSON files saved in:"
)

print(OUTPUT_DIR)